# Seção 3 — Embeddings semânticos e recuperação de informação

## 1. Configuração inicial

In [1]:
import os
import warnings
import logging

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

warnings.filterwarnings("ignore")

logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("torch").setLevel(logging.ERROR)

try:
    from huggingface_hub.utils import logging as hf_logging
    hf_logging.set_verbosity_error()
except Exception:
    pass

In [2]:
from pathlib import Path
import json
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

print("Dispositivo para embeddings:", DEVICE)

Dispositivo para embeddings: cpu


## 2. Caminhos do projeto

In [3]:
CURRENT_DIR = Path.cwd()
ROOT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

DATA_RAW = ROOT_DIR / "data" / "raw"
DATA_PROCESSED = ROOT_DIR / "data" / "processed"
OUTPUT_DIR = ROOT_DIR / "outputs" / "avaliacoes"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Diretório raiz:", ROOT_DIR)
print("Corpus:", DATA_RAW)
print("Arquivos processados:", DATA_PROCESSED)
print("Saídas:", OUTPUT_DIR)

Diretório raiz: C:\Users\breno\Desktop\Sistemas Cognitivos com LLMs - Assessment
Corpus: C:\Users\breno\Desktop\Sistemas Cognitivos com LLMs - Assessment\data\raw
Arquivos processados: C:\Users\breno\Desktop\Sistemas Cognitivos com LLMs - Assessment\data\processed
Saídas: C:\Users\breno\Desktop\Sistemas Cognitivos com LLMs - Assessment\outputs\avaliacoes


## 3. Carregamento do corpus

In [4]:
arquivos = sorted(DATA_RAW.glob("*.txt"))

if not arquivos:
    raise FileNotFoundError(f"Nenhum arquivo .txt encontrado em {DATA_RAW}")

documentos = []
for arquivo in arquivos:
    texto = arquivo.read_text(encoding="utf-8")
    documentos.append({
        "arquivo": arquivo.name,
        "texto": texto
    })

df_docs = pd.DataFrame(documentos)

print("Total de documentos:", len(df_docs))
df_docs[["arquivo"]]

Total de documentos: 12


,arquivo
0,01_contexto_geral_1836_1936.txt
1,02_reino_unido_primeira_industrializacao.txt
2,03_ferrovias_carvao_aco.txt
3,04_segunda_revolucao_industrial.txt
4,05_estados_unidos_capitalismo_industrial.txt
5,06_alemanha_industria_ciencia_estado.txt
6,07_japao_modernizacao_meiji.txt
7,08_imperialismo_materias_primas.txt
8,09_trabalho_urbano_movimentos_operarios.txt
9,10_primeira_guerra_economia_industrial.txt


## 4. Chunking dos documentos

In [5]:
def criar_chunks(texto: str, tamanho: int = 900, sobreposicao: int = 150) -> list[str]:
    texto = " ".join(texto.split())
    chunks = []
    inicio = 0

    while inicio < len(texto):
        fim = inicio + tamanho
        chunk = texto[inicio:fim].strip()

        if chunk:
            chunks.append(chunk)

        if fim >= len(texto):
            break

        inicio = fim - sobreposicao

    return chunks


registros_chunks = []

for doc in documentos:
    chunks = criar_chunks(doc["texto"], tamanho=900, sobreposicao=150)

    for i, chunk in enumerate(chunks, start=1):
        registros_chunks.append({
            "chunk_id": f"{doc['arquivo']}::chunk_{i:03d}",
            "arquivo": doc["arquivo"],
            "chunk_num": i,
            "texto": chunk,
            "tamanho_caracteres": len(chunk)
        })

df_chunks = pd.DataFrame(registros_chunks)

print("Total de chunks:", len(df_chunks))
df_chunks.head()

Total de chunks: 97


,chunk_id,arquivo,chunk_num,texto,tamanho_caracteres
0,01_contexto_geral_1836_1936.txt::chunk_001,01_contexto_geral_1836_1936.txt,1,Título: Contexto geral da industrialização e d...,899
1,01_contexto_geral_1836_1936.txt::chunk_002,01_contexto_geral_1836_1936.txt,2,ternos. O objetivo é apoiar testes de recupera...,900
2,01_contexto_geral_1836_1936.txt::chunk_003,01_contexto_geral_1836_1936.txt,3,"XIX, muitos elementos da primeira industrializ...",900
3,01_contexto_geral_1836_1936.txt::chunk_004,01_contexto_geral_1836_1936.txt,4,o após a Restauração Meiji. A industrialização...,899
4,01_contexto_geral_1836_1936.txt::chunk_005,01_contexto_geral_1836_1936.txt,5,r e expansão colonial. Colônias e regiões peri...,899


## 5. Modelo de embeddings

In [6]:
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

modelo_embeddings = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)

print("Modelo de embeddings carregado:", EMBEDDING_MODEL)

Modelo de embeddings carregado: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


## 6. Geração dos embeddings

In [7]:
textos_chunks = df_chunks["texto"].tolist()

embeddings_chunks = modelo_embeddings.encode(
    textos_chunks,
    normalize_embeddings=True,
    show_progress_bar=False
)

embeddings_chunks = np.array(embeddings_chunks)

print("Formato da matriz de embeddings:", embeddings_chunks.shape)

Formato da matriz de embeddings: (97, 384)


## 7. Salvamento dos chunks processados

In [8]:
chunks_csv = DATA_PROCESSED / "chunks.csv"
df_chunks.to_csv(chunks_csv, index=False, encoding="utf-8")

print("Chunks salvos em:", chunks_csv)

Chunks salvos em: C:\Users\breno\Desktop\Sistemas Cognitivos com LLMs - Assessment\data\processed\chunks.csv


## 8. Função de busca semântica por cosseno

In [9]:
def buscar_chunks(consulta: str, top_k: int = 3) -> pd.DataFrame:
    embedding_consulta = modelo_embeddings.encode(
        [consulta],
        normalize_embeddings=True
    )

    similaridades = cosine_similarity(embedding_consulta, embeddings_chunks)[0]
    indices = np.argsort(similaridades)[::-1][:top_k]

    resultados = df_chunks.iloc[indices].copy()
    resultados["similaridade_cosseno"] = similaridades[indices]
    resultados["consulta"] = consulta

    return resultados[[
        "consulta",
        "chunk_id",
        "arquivo",
        "chunk_num",
        "similaridade_cosseno",
        "texto"
    ]]

## 9. Consultas de teste

In [10]:
consultas_teste = [
    {
        "id": "Q1",
        "consulta": "Como as ferrovias transformaram a economia industrial no século XIX?",
        "expectativa": "Deve recuperar documentos sobre ferrovias, carvão, aço, transporte e integração de mercados."
    },
    {
        "id": "Q2",
        "consulta": "A União Soviética representou uma alternativa ao capitalismo industrial?",
        "expectativa": "Deve recuperar documentos sobre URSS, planejamento estatal e industrialização soviética."
    },
    {
        "id": "Q3",
        "consulta": "Qual foi o papel da América Latina na industrialização entre 1836 e 1936?",
        "expectativa": "Deve revelar limitação do corpus, pois não há documento específico sobre América Latina."
    }
]

todos_resultados = []

for item in consultas_teste:
    resultado = buscar_chunks(item["consulta"], top_k=3)
    resultado.insert(0, "id_consulta", item["id"])
    resultado["expectativa"] = item["expectativa"]
    todos_resultados.append(resultado)

df_resultados = pd.concat(todos_resultados, ignore_index=True)

df_resultados[[
    "id_consulta",
    "consulta",
    "arquivo",
    "chunk_num",
    "similaridade_cosseno"
]]

,id_consulta,consulta,arquivo,chunk_num,similaridade_cosseno
0,Q1,Como as ferrovias transformaram a economia ind...,03_ferrovias_carvao_aco.txt,6,0.803007
1,Q1,Como as ferrovias transformaram a economia ind...,01_contexto_geral_1836_1936.txt,3,0.756804
2,Q1,Como as ferrovias transformaram a economia ind...,03_ferrovias_carvao_aco.txt,2,0.752300
3,Q2,A União Soviética representou uma alternativa ...,12_urss_planejamento_industrializacao.txt,7,0.826304
4,Q2,A União Soviética representou uma alternativa ...,12_urss_planejamento_industrializacao.txt,6,0.826194
5,Q2,A União Soviética representou uma alternativa ...,12_urss_planejamento_industrializacao.txt,5,0.823617
6,Q3,Qual foi o papel da América Latina na industri...,08_imperialismo_materias_primas.txt,2,0.693548
7,Q3,Qual foi o papel da América Latina na industri...,01_contexto_geral_1836_1936.txt,2,0.658027
8,Q3,Qual foi o papel da América Latina na industri...,08_imperialismo_materias_primas.txt,6,0.645503


## 10. Visualização dos trechos recuperados

In [11]:
for item in consultas_teste:
    consulta_id = item["id"]
    consulta = item["consulta"]

    print("=" * 100)
    print(f"{consulta_id} — {consulta}")
    print("Expectativa:", item["expectativa"])
    print("-" * 100)

    subset = df_resultados[df_resultados["id_consulta"] == consulta_id]

    for _, row in subset.iterrows():
        print(f"Arquivo: {row['arquivo']} | chunk {row['chunk_num']} | similaridade: {row['similaridade_cosseno']:.4f}")
        print(row["texto"][:700], "...")
        print("-" * 100)

Q1 — Como as ferrovias transformaram a economia industrial no século XIX?
Expectativa: Deve recuperar documentos sobre ferrovias, carvão, aço, transporte e integração de mercados.
----------------------------------------------------------------------------------------------------
Arquivo: 03_ferrovias_carvao_aco.txt | chunk 6 | similaridade: 0.8030
e administração empresarial. Em muitos lugares, as ferrovias foram algumas das primeiras empresas de grande escala, com burocracias complexas e redes nacionais. Ao mesmo tempo, a expansão ferroviária teve efeitos sociais e ambientais. Ela acelerou a ocupação de territórios, deslocou populações, valorizou terras, intensificou extração mineral e ampliou emissões de carvão. Em contextos coloniais, ferrovias frequentemente foram construídas para extrair matérias-primas e conectá-las a portos, reforçando padrões desiguais de integração econômica. Pontos-chave: - Ferrovias reduziram custos de transporte e integraram mercados. - Carvão foi a princi

## 11. Análise objetiva dos resultados

In [12]:
analises = [
    {
        "id_consulta": "Q1",
        "tipo_resultado": "bem-sucedida",
        "analise": (
            "A consulta sobre ferrovias funcionou bem, pois recuperou documentos diretamente "
            "relacionados a ferrovias, carvão, aço, transporte e integração de mercados. "
            "Esse resultado indica que os embeddings capturaram adequadamente a relação "
            "semântica entre a pergunta e o conteúdo do corpus."
        )
    },
    {
        "id_consulta": "Q2",
        "tipo_resultado": "bem-sucedida",
        "analise": (
            "A consulta sobre a União Soviética funcionou bem, pois recuperou o documento "
            "sobre planejamento e industrialização soviética. A pergunta é bem delimitada "
            "e possui correspondência direta com um dos textos do corpus."
        )
    },
    {
        "id_consulta": "Q3",
        "tipo_resultado": "limitação do corpus",
        "analise": (
            "A consulta sobre América Latina recuperou trechos relacionados a imperialismo, "
            "matérias-primas e industrialização geral, mas o corpus não possui documento "
            "específico sobre América Latina. Isso indica que a busca encontrou contexto "
            "semanticamente próximo, porém insuficiente para responder diretamente à pergunta. "
            "Em um pipeline RAG, essa limitação deve ser informada ao usuário."
        )
    }
]

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 2000)

df_analise = pd.DataFrame(analises)

df_analise.style.set_properties(
    subset=["analise"],
    **{
        "white-space": "pre-wrap",
        "text-align": "left"
    }
)

,id_consulta,tipo_resultado,analise
0,Q1,bem-sucedida,"A consulta sobre ferrovias funcionou bem, pois recuperou documentos diretamente relacionados a ferrovias, carvão, aço, transporte e integração de mercados. Esse resultado indica que os embeddings capturaram adequadamente a relação semântica entre a pergunta e o conteúdo do corpus."
1,Q2,bem-sucedida,"A consulta sobre a União Soviética funcionou bem, pois recuperou o documento sobre planejamento e industrialização soviética. A pergunta é bem delimitada e possui correspondência direta com um dos textos do corpus."
2,Q3,limitação do corpus,"A consulta sobre América Latina recuperou trechos relacionados a imperialismo, matérias-primas e industrialização geral, mas o corpus não possui documento específico sobre América Latina. Isso indica que a busca encontrou contexto semanticamente próximo, porém insuficiente para responder diretamente à pergunta. Em um pipeline RAG, essa limitação deve ser informada ao usuário."


## 12. Salvamento dos resultados

In [13]:
resultados_csv = OUTPUT_DIR / "c03_resultados_busca.csv"
analise_csv = OUTPUT_DIR / "c03_analise_busca.csv"
resultados_json = OUTPUT_DIR / "c03_embeddings_busca_resultados.json"

df_resultados.to_csv(resultados_csv, index=False, encoding="utf-8")
df_analise.to_csv(analise_csv, index=False, encoding="utf-8")

saida_json = {
    "data_execucao": datetime.now().isoformat(),
    "modelo_embeddings": EMBEDDING_MODEL,
    "device": DEVICE,
    "total_documentos": int(len(df_docs)),
    "total_chunks": int(len(df_chunks)),
    "estrategia_recuperacao": "Embeddings sentence-transformers + similaridade de cosseno",
    "top_k": 3,
    "consultas_teste": consultas_teste,
    "analise": df_analise.to_dict(orient="records")
}

with resultados_json.open("w", encoding="utf-8") as f:
    json.dump(saida_json, f, ensure_ascii=False, indent=2)

print("Resultados salvos em:")
print("-", resultados_csv)
print("-", analise_csv)
print("-", resultados_json)

Resultados salvos em:
- C:\Users\breno\Desktop\Sistemas Cognitivos com LLMs - Assessment\outputs\avaliacoes\c03_resultados_busca.csv
- C:\Users\breno\Desktop\Sistemas Cognitivos com LLMs - Assessment\outputs\avaliacoes\c03_analise_busca.csv
- C:\Users\breno\Desktop\Sistemas Cognitivos com LLMs - Assessment\outputs\avaliacoes\c03_embeddings_busca_resultados.json
